In [ ]:
import jax
from jax import lax, Array
import jax.numpy as jnp
from jax.scipy.linalg import expm
jax.config.update('jax_enable_x64', True)
import numpy as np
from numpy.linalg import matrix_power
from scipy.stats import entropy, poisson
from scipy.optimize import minimize
from scipy.special import softmax, logsumexp
from itertools import product as iproduct
from time import perf_counter
from functools import partial

import os, sys, pathlib
os.environ["OPENBLAS_NUM_THREADS"] = "1"
_project_root = pathlib.Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[0]))
sys.path.insert(0, str(_project_root.parents[1]))
from physics.markov import H, KL, get_traj
import physics.glauber as ising

GraphT = tuple[Array, Array, Array]

For forward and reverse fluxes $J_e^\pm$, and for the arithmetic mean, dynamical activity defined to be
$$
2\mu_A(t) = \sum_e (J_e^+(t) + J_e^-(t)),
$$
which is the expected number of jumps per unit time

Escape rate from $\sigma$:
$$
\lambda(\sigma) = -K_{\sigma\sigma},
$$
so expected number of jumps per unit time:
$$
\sum_\sigma p_t(\sigma)\lambda(\sigma).
$$
Then,
$$
\mathcal{A}_\tau = \int_{[0,\tau]}\lambda^\top p_t dt
$$
is the paper's
$$
\mathcal{A}_\tau = \int_{[0,\tau]}2 \mu_A(t) dt
$$

Activity, which represents the kinetic property of dynamics, plays a central role in obtaining thermodynamic
speed limits (TSLs)

In [ ]:
# ── Parameters ──────────────────────────────
L      = 2
shape  = (L, L)
N      = L * L
n_states = 1 << N

# ── Initial condition ─────────────────────────────────────────────────────────
# delta function — all spins up (larger KL, more interesting dynamics)
all_up_idx = (1 << N) - 1
p0 = jnp.zeros(n_states)
p0 = p0.at[all_up_idx].set(1.0)
# p0 = jnp.ones((1 << N)) / (1 << N)

J      = 0.2
h      = 0.0
T1, T2 = 1/3.0, 1/1.0
betas_jax = jnp.array([1/T1, 1/T2])
gammas_nu = jnp.array([1.0, 1.0])
# gammas_nu = jnp.array([0.5, 0.5])
dt = 0.1

# critical temperature
T_c = 2 * J / np.log(1 + np.sqrt(2))

# speed limit eps
eps = 0.05

# graph = ising.make_interaction_graph(shape)
graph = ising.make_mean_field_graph(N)

In [ ]:
N_max = 200
rows, rates, rates_nu, escape, g, lam, w, d = ising.setup_uniformization(
  graph, J, h, betas_jax, gammas_nu, dt,
)

pN, pts, heat_steps, activity_steps = ising.get_uniformized_traj(
  p0, rows, rates, g, lam, w, d, N_max,
)

In [17]:

rows = graph[0]
cols = jnp.tile(jnp.arange(n_states), N).reshape(*rows.shape)
mask_u = rows < cols
mask_l = rows > cols

reverse_rates = jnp.stack([
  jnp.take_along_axis(rates_nu[0], rows, axis=1)[mask_u],
  jnp.take_along_axis(rates_nu[1], rows, axis=1)[mask_u]
], axis=0)

# rates2 = jnp.stack([
#   jnp.take_along_axis(rates_nu[0], cols, axis=1)[mask_u],
#   jnp.take_along_axis(rates_nu[1], cols, axis=1)[mask_u]
# ], axis=0)

masked_rates = rates_nu[:, mask_u]
# this is same as reverse_rates
masked_rates2 = rates_nu[:, mask_l]

# ed/ge_dst = rows[mask_u]
# edge_src = cols[mask_u]

gather_indices_u = cols[mask_u].reshape(-1)[:, None]
# want the probability at the destination of the forward edge, i.e. rows[mask_u], not cols[mask_l]
gather_indices_l = cols[mask_l].reshape(-1)[:, None]

dimension_numbers = lax.GatherDimensionNumbers(
  offset_dims=(),
  collapsed_slice_dims=(0,),
  start_index_map=(0,)
)

def mu_A(p):
  Jp = masked_rates * lax.gather(p, gather_indices_u, dimension_numbers, slice_sizes=(1,))[None, :]
  Jm = reverse_rates * lax.gather(p, gather_indices_l, dimension_numbers, slice_sizes=(1,))[None, :]
  return jnp.sum(0.5 * (Jp + Jm))

# exactly the same
# print(rates_nu[0][mask_l] - reverse_rates[0])
# print(rates2[0] - rates_nu[0][mask_u])


mu_A_vals = jax.vmap(mu_A)(pts)


### Comparison (dense)

In [ ]:
rates_nu = get_rates(graph, J, h, betas_jax, gammas_nu)
rows = graph[0]
K1 = ising.dense_generator_from_rates(rows, rates_nu[0])
K2 = ising.dense_generator_from_rates(rows, rates_nu[1])


In [19]:
import math
print(jnp.sum((K1 + K1.T > 0).astype(jnp.int32)))
print(jnp.sum((jnp.triu(K1 + K1.T, 1) >0).astype(jnp.int32)))
print(jnp.sum(jnp.diag(K1 + K1.T) > 0))
print(math.prod(rows.shape))


64
32
0
64


In [20]:
def _edge_indices(K_nu):
  K_sym = K_nu + K_nu.T
  idx = np.argwhere(np.triu(K_sym, 1) > 0)
  return idx[:, 0], idx[:, 1]
# print((np.triu(K1, 1) > 0).astype(np.uint))

# all give the same -> K1, K2 both symmetric and have same edges
r1, c1 = _edge_indices(K1)
# r1, c1 = _edge_indices(np.triu(K1, 1))
r2, c2 = _edge_indices(K2)

def mu_A(p):
  Jp = np.concatenate([K1[r1, c1]*p[c1], K2[r2, c2]*p[c2]]) # do the diff reservoirs separately, then add
  Jm = np.concatenate([K1[c1, r1]*p[r1], K2[c2, r2]*p[r2]])
  return np.sum(0.5*(Jp + Jm)) #arithemetic meannn

pts_np = np.asarray(pts)
mu_A_vals = np.array([mu_A(p) for p in pts_np])
# 

import ot

states    = [np.array(s, dtype=np.int8) for s in iproduct([-1, 1], repeat=N)]
states_arr = np.array(states)
D = np.asarray((states_arr[:, None] != states_arr[None, :]).sum(axis=2).astype(float))

W1          = np.zeros(N_max)
mu_L_cum    = np.zeros(N_max)
speed_limit = np.zeros(N_max)


p_start = pts_np[0]

for n in range(N_max):
  p_end = pts_np[n+1]

  w1 = ot.emd2(p_start, p_end, D) # i think this is same as W1 in the paper since we only flip one spin

  T_n = (n+1) * dt
  seg = mu_A_vals[:n+2]
  mu_A_avg = np.trapezoid(seg) / (n+1)   # average activity over [0, T_n]

  v1    = w1 / T_n                  
  ratio = np.clip(v1 / (2 * mu_A_avg), 0, 1 - 1e-9)

  speed_limit[n] = 2 * w1 * np.arctanh(ratio)   # note: w1, not v1

print(speed_limit)

[1.10878476 1.38799579 1.45619622 1.43341358 1.36910285 1.2873289
 1.20055014 1.11525983 1.03470332 0.96031654 0.89252904 0.83122468
 0.77600863 0.72636156 0.68172718 0.6415603  0.60535139 0.57263757
 0.54300563 0.5160909  0.49157377 0.4691754  0.44865307 0.42979572
 0.41241978 0.39636552 0.38149372 0.36768282 0.35482647 0.34283141
 0.33161563 0.32110686 0.31124122 0.30196209 0.29321917 0.28496763
 0.27716742 0.26978269 0.26278123 0.25613403 0.24981493 0.24380026
 0.23806857 0.23260037 0.22737789 0.22238495 0.21760674 0.21302969
 0.20864136 0.20443031 0.20038601 0.19649874 0.19275953 0.18916006
 0.18569265 0.18235015 0.17912593 0.17601382 0.17300806 0.1701033
 0.16729453 0.16457705 0.1619465  0.15939876 0.15692998 0.15453654
 0.15221505 0.14996231 0.1477753  0.1456512  0.14358732 0.14158113
 0.13963026 0.13773244 0.13588553 0.13408752 0.13233649 0.13063062
 0.12896818 0.12734754 0.12576714 0.12422549 0.12272119 0.1212529
 0.11981934 0.11841929 0.1170516  0.11571514 0.11440886 0.1131317

In [21]:
cols = jnp.tile(jnp.arange(n_states), N)
# print(idxs)
mask_u = rows.reshape(-1) < cols
idxs = jnp.stack([rows.reshape(-1), cols], axis=-1)[mask_u]

order = jnp.lexsort([cols[mask_u], rows.reshape(-1)[mask_u]], axis=0)
print(idxs[order][:, 0])
print(idxs[order][:, 1])

[ 0  0  0  0  1  1  1  2  2  2  3  3  4  4  4  5  5  6  6  7  8  8  8  9
  9 10 10 11 12 12 13 14]
[ 1  2  4  8  3  5  9  3  6 10  7 11  5  6 12  7 13  7 14 15  9 10 12 11
 13 11 14 15 13 14 15 15]


In [22]:
cols = jnp.tile(jnp.arange(n_states), N).reshape(*rows.shape)
# print(idxs)
mask_u = rows < cols
idxs = jnp.stack([rows, cols], axis=-1)[mask_u]

order = jnp.lexsort([cols[mask_u], rows[mask_u]], axis=0)
print(idxs[order][:, 0])
print(idxs[order][:, 1])

[ 0  0  0  0  1  1  1  2  2  2  3  3  4  4  4  5  5  6  6  7  8  8  8  9
  9 10 10 11 12 12 13 14]
[ 1  2  4  8  3  5  9  3  6 10  7 11  5  6 12  7 13  7 14 15  9 10 12 11
 13 11 14 15 13 14 15 15]
